# 2.0 — Procesamiento: temperatura SMN/CONAGUA

Convierte los resúmenes mensuales por
estación de `01_ingesta/1.0-conagua-temperatura.ipynb` (TMed, TMax, TMin)
en una sola serie mensual para Hermosillo, la que usa el panel integrado
(`03_eda/3.0-integracion-espacial.ipynb`).

## Entradas

| Ruta | Contenido |
|------|-----------|
| `data/raw/temperatura/temperatura_AAAA/*.csv` | Temperatura media mensual por estación (`Tmed`) |
| `data/raw/temperatura/temperatura_max_AAAA/*.csv` | Temperatura máxima mensual por estación (`Tmax`) |
| `data/raw/temperatura/temperatura_min_AAAA/*.csv` | Temperatura mínima mensual por estación (`Tmin`) |

## Salidas

- `data/interim/clima/temperatura_smn_hermosillo_mensual.csv` (opcional,
  para revisión): `fecha_mes`, `temp_media_c`, `temp_max_media_c`,
  `temp_min_media_c`.
- Decisiones de limpieza registradas al final del notebook.

## Funciones del paquete

`panel.load_smn_temperature()`, `load_smn_temperature_max()`,
`load_smn_temperature_min()` y `panel.climate_hermosillo()` (filtra las
estaciones de `config.CLIMATE_STATION_IDS`, promedia y marca los meses
corruptos).

## Checklist

- [x] Cobertura: meses con dato por estación de Hermosillo.
- [x] Valores centinela del SMN (`-99.9`, `-999`) y rangos físicos (`Tmin ≤ Tmed ≤ Tmax`).
- [x] Meses corruptos en la fuente (mismo valor en TMed/TMax/TMin) → NaN.
- [x] Promedio entre estaciones y número de estaciones que aporta cada mes.
- [x] Faltantes en la serie mensual final (se tratan en `03_eda/3.1-datos-faltantes.ipynb`).


### ¿Qué problema resuelve este notebook?

Los archivos del SMN traen la temperatura de **más de 1,900 estaciones de todo México**, mes por mes. Pero el proyecto necesita **una sola serie de temperatura para Hermosillo** (un valor por mes), y que sea **confiable**.

Este notebook es una **revisión de calidad**: antes de usar los datos, comprobamos que no tengan errores.

### El recorrido, paso a paso

```
Configuración  →  Carga  →  Cobertura  →  Centinelas y rangos  →  Meses corruptos  →  Serie de Hermosillo  →  Guardar
(herramientas)   (leer     (¿hay datos    (¿hay valores          (¿hay meses         (promedio mensual     (opcional)
                  datos)    suficientes?)  imposibles?)           dañados?)           + gráfica)
```


### Mini glosario

| Término | Qué significa |
|---------|---------------|
| **Estación** | Un punto de medición del clima. Se identifica con su `clave` (por ejemplo `HM2SO`). |
| **Estación-mes** | Una fila de la tabla: la lectura de **una estación en un mes**. |
| **`NaN`** | Un valor vacío (sin dato). |
| **Valor centinela** | Un número como `-99.9` que la fuente usa para decir "no hay dato". Si se lee como temperatura real, arruina los promedios. |
| **`fecha_mes`** | El mes de la lectura (por ejemplo `2015-01`). |
| **Filtrar** | Quedarse solo con las filas que cumplen una condición. |
| **`groupby` (agrupar)** | Juntar filas que comparten algo (por ejemplo, el mismo mes) para contarlas o promediarlas. |
| **`merge` (unir)** | Pegar dos tablas usando columnas en común, como buscar con `BUSCARV` en Excel. |

## Configuración e imports

En esta parte se cargan las herramientas (librerías) y las funciones del proyecto que se usarán en el resto del notebook.

**Qué trae cada parte:**
- `CLIMATE_STATION_IDS` (de `config.py`): las claves de las estaciones de Hermosillo.
- Las funciones de `panel.py`: leen los CSV del SMN y calculan la serie de Hermosillo.

**Cómo leer el resultado:** debe imprimir `Estaciones de Hermosillo: ['HM2SO', 'HMNSO', 'MMHO']`.

**Si falla:** el mensaje `No module named 'sonora_health_climate_eda'` significa que el kernel elegido no es el del proyecto, o falta instalarlo con `pip install -e .`.

In [1]:
from pathlib import Path
import sys

ROOT_DIR = Path.cwd().parent.parent  # notebooks/<seccion>/ -> raiz del proyecto
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sonora_health_climate_eda.config import CLIMATE_STATION_IDS, INTERIM_DATA_DIR, PROCESSED_DATA_DIR
from sonora_health_climate_eda.panel import (
    climate_hermosillo,        # promedia las estaciones de Hermosillo por mes
    load_smn_temperature,      # temperatura media de todas las estaciones
    load_smn_temperature_max,  # temperatura máxima de todas las estaciones
    load_smn_temperature_min,  # temperatura mínima de todas las estaciones
)

OUT_DIR = INTERIM_DATA_DIR / "clima"
print("Estaciones de Hermosillo:", CLIMATE_STATION_IDS)


Estaciones de Hermosillo: ['HM2SO', 'HMNSO', 'MMHO']


## Carga por estación

En esta parte se leen todos los CSV mensuales del SMN (los que están en `data/raw/`) y se juntan en tres tablas: temperatura media (`tmed`), máxima (`tmax`) y mínima (`tmin`). Se busca tener los datos en tablas con las que se pueda trabajar. Cada fila es **una estación en un mes**, de todo México. Para cada tabla se imprime cuántas filas tiene, cuántas estaciones distintas hay y de qué mes a qué mes van los datos. Debe verse `2015-01 a 2020-12`.

**Resultado de nuestra corrida:** cada tabla tiene unas 77,000 filas y 1,939 estaciones, con datos de 2015-01 a 2020-12. Son 72 meses con unas 1,070 estaciones por mes. Las tres tablas tienen un número casi igual de filas, y la pequeña diferencia es normal: no todas las estaciones reportan las tres temperaturas cada mes.

**Si falla:** el error `No objects to concatenate` significa que faltan las carpetas de temperatura máxima o mínima en `data/raw/` (se descargan con `download-temperature-extremes-data`).

In [ ]:
# cada función lee todos los CSV mensuales de su carpeta en data/raw/ y los junta
tmed = load_smn_temperature()
tmax = load_smn_temperature_max()
tmin = load_smn_temperature_min()

for nombre, df in {"TMed": tmed, "TMax": tmax, "TMin": tmin}.items():
    print(
        f"{nombre}: {len(df):,} filas estación-mes, "
        f"{df['clave'].nunique()} estaciones, "
        f"{df['fecha_mes'].min()} a {df['fecha_mes'].max()}"
    )


## Cobertura de las estaciones de Hermosillo

Ahora se cuenta, para cada estación de Hermosillo, en cuántos meses tiene dato de temperatura media. Después se cuenta, para cada mes, cuántas de las 3 estaciones tienen dato.

**Para qué sirve:** la serie de Hermosillo es el **promedio de sus estaciones**. Si una estación casi no tiene datos, el promedio mezcla distinta cantidad de estaciones según el mes, y eso puede crear saltos que no son clima. Un mes sin ninguna estación queda como `NaN` después del promedio.

**Cómo leer el resultado:**
- Primera tabla: meses con dato por estación. El máximo posible es 72 (de 2015 a 2020).
- Segunda tabla: cuántos meses tienen 1, 2 o 3 estaciones con dato.
- Si una estación no aparece, su clave en `config.py` podría estar mal escrita.

**Resultado de nuestra corrida:**

| Estación | Meses con dato (de 72) |
|----------|------------------------|
| HM2SO | 72 (completa) |
| HMNSO | 70 |
| MMHO | 36 (solo la mitad) |

36 meses tienen las 3 estaciones, 34 tienen 2 y solo 2 tienen 1. La suma es 72, así que **ningún mes quedó sin datos**. Lo que sí conviene tener presente es que la calidad del promedio cambia entre meses (por eso el `TODO` del final sobre agregar `n_estaciones`).

In [ ]:
herm = tmed[tmed["clave"].isin(CLIMATE_STATION_IDS)]
cobertura = (
    herm.dropna(subset=["temp_media_c"])
    .groupby(["fecha_mes", "clave"])
    .size()
    .unstack("clave")
)
print("Meses con dato por estación (TMed):")
print(cobertura.notna().sum())

# cuántas estaciones aportan dato en cada mes
n_estaciones = cobertura.notna().sum(axis=1)
print("\nMeses según número de estaciones con dato:")
print(n_estaciones.value_counts().sort_index())


## Valores centinela y rangos físicos

**Qué se hace:** dos revisiones.
1. **Centinelas:** se cuenta cuántas filas traen valores como `-99.9`, `-999` o `-99`. Son códigos que significan "sin dato"; si se leen como temperatura real, destruyen cualquier promedio.
2. **Rangos físicos:** se unen las tres tablas (media, máxima, mínima) y se buscan filas imposibles. Siempre debe cumplirse **mínima ≤ media ≤ máxima**.

**Para qué sirve:** detectar datos imposibles antes de que lleguen a la serie de Hermosillo.

**Cómo leer el resultado:**
- `centinelas=0` es lo ideal. Si es mayor, hay que ver si esas filas son de Hermosillo (para eso sirve la celda de verificación que sigue).
- `NaN` es la cantidad de filas vacías.
- La tabla del final muestra los casos incoherentes **de Hermosillo**. Si sale vacía (`Empty DataFrame`), no hay problemas ahí.

**Resultado de nuestra corrida:**
- Hay **1 centinela** (`-99.9`) en cada serie, a nivel de todo México. Aún no sabemos de qué estación es.
- Hay 66 filas vacías en cada serie (el mismo número, así que probablemente son las mismas filas).
- Hay 13 casos incoherentes en todo el país, pero la tabla de Hermosillo salió **vacía**: ninguno es de nuestras estaciones.
- El máximo de la temperatura mínima (38.0) es sospechoso, porque una mínima mensual promedio de 38 °C es casi imposible. Puede venir de los meses dañados que se revisan más abajo. Lo importante es que la gráfica final de Hermosillo no muestra valores raros.

In [ ]:
SERIES = [("TMed", tmed, "temp_media_c"), ("TMax", tmax, "temp_max_media_c"), ("TMin", tmin, "temp_min_media_c")]
for nombre, df, col in SERIES:
    # valores centinela del SMN (-99.9, -999, -99) que no son temperaturas reales
    n_centinela = df[col].isin([-99.9, -999.0, -99.0]).sum()
    print(
        f"{nombre}: centinelas={n_centinela}, min={df[col].min():.1f}, max={df[col].max():.1f}, NaN={df[col].isna().sum()}"
    )

# coherencia física: no puede pasar que Tmin > Tmed o Tmed > Tmax
llave = ["clave", "fecha_mes"]
est = (
    tmed[llave + ["estacion", "temp_media_c"]]
    .merge(tmax[llave + ["temp_max_media_c"]], on=llave, how="inner")
    .merge(tmin[llave + ["temp_min_media_c"]], on=llave, how="inner")
)
incoherentes = est[
    (est["temp_min_media_c"] > est["temp_media_c"]) | (est["temp_media_c"] > est["temp_max_media_c"])
]
print(f"\nEstación-mes con Tmin > Tmed o Tmed > Tmax: {len(incoherentes)}")
incoherentes[incoherentes["clave"].isin(CLIMATE_STATION_IDS)].head(20)


### Verificación: ¿de dónde vienen los centinelas?

**Qué se hace:** se muestran las filas exactas que traen un valor centinela, con su estación y su mes, y se cuenta si alguna es de Hermosillo.

**Para qué sirve:** la celda anterior solo dijo que hay 1 centinela en cada serie, pero no de quién. Si fuera de una estación de Hermosillo, bajaría mucho el promedio de ese mes. Esta celda lo confirma o lo descarta.

**Cómo leer el resultado:**
- La línea `De Hermosillo: 0` significa que **no afecta** a nuestra serie.
- Si dice un número mayor que 0, hay que corregirlo en `panel.py` convirtiendo esos valores en `NaN` antes de promediar.

In [ ]:
for nombre, df, col in SERIES:
    filas_centinela = df[df[col].isin([-99.9, -999.0, -99.0])]
    print(f"--- {nombre}: filas con valor centinela ---")
    print(filas_centinela[["clave", "estacion", "fecha_mes", col]])
    en_hermosillo = filas_centinela["clave"].isin(CLIMATE_STATION_IDS).sum()
    print(f"De Hermosillo: {en_hermosillo}\n")


## Meses corruptos en la fuente

**Qué se hace:** se buscan meses en los que, para muchas estaciones, la temperatura media, la máxima y la mínima son **idénticas**.

**Para qué sirve:** en la realidad, la máxima de un mes siempre es mayor que la media, y la media mayor que la mínima. Si las tres son iguales, el archivo del SMN está dañado. Se sabe que pasó en dic-2015, dic-2016 y dic-2017: cada estación trae el mismo valor en TMed, TMax y TMin, y ~8–10 °C por encima del diciembre típico. `climate_hermosillo()` ya los marca como `NaN`. Esta celda busca **otros** meses con el mismo problema.

**Cómo leer el resultado:** la lista muestra, por mes, cuántas estaciones tienen las tres temperaturas iguales. Los meses dañados salen con cientos de estaciones.

**Resultado de nuestra corrida:**
- dic-2015: 922 estaciones, dic-2017: 909 y dic-2016: 903. Son justo los tres meses ya conocidos. **No aparece ningún mes dañado nuevo.**
- También aparece `2020-09` con valor 0. Una explicación probable es que corresponda a la fila del valor centinela (`-99.9` en las tres series), que no tiene clave de estación y por eso cuenta 0 estaciones. La celda de verificación de arriba ayuda a confirmarlo. En cualquier caso, 0 significa que no hay estaciones afectadas ese mes.

In [ ]:
# los tres valores iguales a la vez es imposible en la realidad
mismo_valor = est[
    np.isclose(est["temp_media_c"], est["temp_max_media_c"])
    & np.isclose(est["temp_media_c"], est["temp_min_media_c"])
]
print("Meses con más estaciones con TMed = TMax = TMin:")
mismo_valor.groupby("fecha_mes")["clave"].nunique().sort_values(ascending=False).head(10)

# si aparecen meses nuevos con este patrón, agregarlos a `bad_months`
# en panel.climate_hermosillo() y registrarlo en la tabla de decisiones


## Serie mensual de Hermosillo

**Qué se hace:** se ejecuta `climate_hermosillo()`, la función de `panel.py` que produce el resultado final del notebook. Internamente: se queda con las estaciones de Hermosillo, **promedia** sus temperaturas por mes y pone `NaN` en los meses dañados (dic-2015, 2016 y 2017).

**Para qué sirve:** es **la serie que usa el panel** del proyecto: un valor de temperatura media, máxima y mínima por cada mes de Hermosillo.

**Cómo leer el resultado:**
- `isna().sum()` cuenta los meses vacíos por columna. Se esperan 3 (los diciembres dañados).
- `head()` muestra los primeros 5 meses.

**Resultado de nuestra corrida:**
- La serie tiene **72 meses** (ene-2015 a dic-2020), con 3 vacíos en cada columna, tal como se esperaba.
- Los primeros valores son coherentes con Hermosillo: en enero-2015, media 18.8 °C, máxima 26.1 °C y mínima 11.4 °C.
- La función reporta que usa 7 nombres de estación distintos (por ejemplo `HERMOSILLO` y `Observatorio de Hermosillo, Son.`). Son las mismas 3 claves con nombres que cambian entre archivos. Por eso se filtra por `clave` y no por nombre.

In [ ]:
# filtra las estaciones de Hermosillo, promedia por mes y deja vacíos los meses dañados
temp_mensual = climate_hermosillo()
print(temp_mensual.isna().sum())  # se esperan 3: los diciembres dañados
temp_mensual.head()


### Gráfica de la serie

**Qué se hace:** se dibujan las tres temperaturas (mínima, media y máxima) a lo largo del tiempo.

**Para qué sirve:** es la forma más rápida de detectar problemas a simple vista: saltos raros, valores imposibles o huecos inesperados.

**Cómo leer el resultado:** debe verse una **ola** que sube en verano y baja en invierno, con la máxima arriba, la media en medio y la mínima abajo, sin que las líneas se crucen.

**Resultado de nuestra corrida:** la curva es coherente: máximas de ~40–42 °C en verano y mínimas de ~8–12 °C en invierno. Hay **huecos** en los tres diciembres dañados (2015, 2016 y 2017), como se esperaba. No hay caídas bruscas, así que ningún valor centinela afectó la serie.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for col in ["temp_min_media_c", "temp_media_c", "temp_max_media_c"]:
    ax.plot(temp_mensual["fecha_mes"], temp_mensual[col], label=col)
ax.set_ylabel("°C")
ax.set_title("Temperatura mensual SMN — Hermosillo (promedio de estaciones)")
ax.legend()
plt.tight_layout()
plt.show()

# meses promediados con menos de 3 estaciones
n_estaciones[n_estaciones < 3]


## Guardar salida

**Qué se hace:** se guarda la serie de Hermosillo en un archivo CSV, `data/interim/clima/temperatura_smn_hermosillo_mensual.csv`.

**Para qué sirve:** dejar una copia para revisarla fuera del notebook (por ejemplo en Excel). El panel **no** lee el CSV: lee la copia `data/processed/climate_hermosillo.parquet` que guarda la misma celda.

**Importante:** hay que ejecutar esta celda antes del notebook `03_eda/3.0`, porque el panel lee el parquet.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
temp_mensual.to_csv(OUT_DIR / "temperatura_smn_hermosillo_mensual.csv", index=False)
# el panel lee este parquet (build_panel_monthly)
temp_mensual.to_parquet(PROCESSED_DATA_DIR / "climate_hermosillo.parquet", index=False)


## Decisiones de procesamiento

Registrar cada decisión que cambie los datos y su justificación. Si la
decisión cambia el panel, implementarla en `sonora_health_climate_eda/panel.py`
(es lo que ejecuta `03_eda/3.0-integracion-espacial.ipynb`) y llevarla al
reporte final.

| Variable / periodo | Problema | Decisión | Justificación |
|--------------------|----------|----------|---------------|
| dic-2015, dic-2016, dic-2017 (TMed/TMax/TMin) | Archivo SMN degenerado | NaN | Valor idéntico en las tres series y ~8–10 °C sobre el diciembre típico |
| Estaciones HMNSO (70 meses) y MMHO (36 meses), 2015-2020 | Cobertura incompleta: 34 meses con 2 estaciones y 2 meses con 1 | Sin cambios: se promedian las estaciones disponibles de cada mes | Ningún mes queda sin estaciones; pendiente evaluar `n_estaciones` si se detectan saltos |
| TMed/TMax/TMin, rangos físicos | 13 casos con Tmin > Tmed o Tmed > Tmax en todo el país | Sin cambios | Ninguno corresponde a estaciones de Hermosillo (tabla vacía) |
| Otros meses con TMed = TMax = TMin | Se buscó el patrón de archivo dañado en todos los meses | Sin cambios | Solo aparecen dic-2015, dic-2016 y dic-2017, ya tratados como NaN |
| TMed/TMax/TMin, sep-2020 (1 fila por serie con valor `-99.9`) | Fila con valor centinela y sin clave ni nombre de estación | Sin cambios | No corresponde a ninguna estación de Hermosillo, así que no entra al promedio de la serie |

In [ ]:
temp_mensual.head()
